##**Projeto:** Merca Data Platform

##**Squad:** 2 | Camada Gold
### Objetivo
Promover os dados validados do catálogo de produtos da Silver para a camada Gold, enriquecendo com alertas de qualidade de negócio e disponibilizando para o SQL Server.
### Origem e Destinos
| Item | Valor |
| **Origem** | `squad2/silver/ecommerce_produtos` (Delta Lake) |
| **Destino Lake** | `squad2/gold/ecommerce_produtos` (Delta Lake — append) |
| **Destino SQL** | `squad2.gold_ecommerce_produtos` (SQL Server — append) |
| **Controle** | `gold/control/ecommerce_produtos.json` |
### Regras de Negócio Aplicadas
| # | Regra | Descrição | Saída |
| 4 | Alerta de volume anormal | Mais de 50 SKUs novos em um lote sugere carga de teste | Log de alerta |
| 5 | Produto ativo com preço zerado | Produto com `is_ativo = True` e `preco_lista <= 0` | Campo `alerta_produto_ativo_preco_zero` |
### Campos Calculados na Gold
| Campo | Lógica | Valores |
| `alerta_produto_ativo_preco_zero` | Produto ativo com preço inválido | `'SIM'` / `'NAO'` |
| `gold_processed_at` | Timestamp de processamento na Gold | datetime |
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | `get_storage_options`, `get_squad2_client`, `SQL_OPTIONS` |
| `feat_squad2_silver_produtos` | Dados validados de origem |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
from deltalake import DeltaTable, write_deltalake
import pandas as pd
from datetime import datetime
import json

TABELA = "ecommerce_produtos"
TABELA_SQL = f"gold_{TABELA}"  

path_silver  = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA}"
path_gold    = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/gold/{TABELA}"
path_control = f"gold/control/{TABELA}.json"

- Processamento e Enriquecimento para a Gold
 **Controle incremental:** apenas arquivos ainda não processados pela Gold são considerados.
**Regra de Negócio 5 — Alerta de Produto Ativo com Preço Zerado:**
Produtos com `is_ativo = True` e `preco_lista <= 0` recebem `alerta_produto_ativo_preco_zero = 'SIM'`.
Este alerta sinaliza erros de cadastro que podem impactar vendas online.
**Sink duplo:** dados gravados simultaneamente no Delta Lake e no SQL Server.
 O conector SQL cria a tabela automaticamente na primeira execução ou faz append alinhando as colunas.


In [0]:
try:
    if not DeltaTable.is_deltatable(path_silver, storage_options=get_storage_options()):
        print(f" [Aviso Gold] A tabela Silver de {TABELA} ainda não foi inicializada.")
    else:
        dt_silver = DeltaTable(path_silver, storage_options=get_storage_options())
        df_pandas = dt_silver.to_pandas()
        
        squad2_client = get_squad2_client()
        file_client = squad2_client.get_file_client(path_control)
        
        processados = set()
        if file_client.exists():
            conteudo = file_client.download_file().readall().decode('utf-8')
            processados = set(json.loads(conteudo))
        
        df_novos_dados = df_pandas[~df_pandas['bronze_source_file'].isin(processados)].copy()
        #df_novos_dados = df_pandas.copy() --forçar dados manualmente
        
        if df_novos_dados.empty:
            print(" Camada Gold de Produtos em dia! Nenhum dado novo para processar.")
        else:
            print(f" Processando {len(df_novos_dados)} linhas para a Tabela Final...")
            
            # Regra de Negócio: Alerta de Preço Bugado
            df_novos_dados['alerta_produto_ativo_preco_zero'] = df_novos_dados.apply(
                lambda row: 'SIM' if (row['is_ativo'] == True and row['preco_lista'] <= 0) else 'NAO', axis=1
            )
            
            #  Coluna de Auditoria e Verificação de Atualização
            df_novos_dados['gold_processed_at'] = datetime.now()
            
            # Limpeza preventiva de fuso horário para o motor Spark/Delta
            for col in df_novos_dados.columns:
                if pd.api.types.is_datetime64_any_dtype(df_novos_dados[col]):
                    df_novos_dados[col] = df_novos_dados[col].dt.tz_localize(None)
            
            # SINK 1: Gravação Lakehouse (Pasta Gold)
            write_deltalake(path_gold, df_novos_dados, mode="append", storage_options=get_storage_options())
            
            # -------------------------------------------------------------------------
            # SINK 2: INGESTÃO NO SQL SERVER COM CRIAÇÃO E ALINHAMENTO AUTOMÁTICO
            # -------------------------------------------------------------------------
            try:
                # Se a tabela já existir, lê a estrutura para alinhar ordem e case das colunas
                df_schema_sql = spark.read \
                    .format("sqlserver") \
                    .options(**SQL_OPTIONS) \
                    .option("dbtable", f"[squad2].[{TABELA_SQL}]") \
                    .load() \
                    .limit(0)
                
                spark_df_final = spark.createDataFrame(df_novos_dados)
                for col_db in df_schema_sql.columns:
                    col_match = [c for c in spark_df_final.columns if c.lower() == col_db.lower()]
                    if col_match:
                        spark_df_final = spark_df_final.withColumnRenamed(col_match[0], col_db)
                spark_df_aligned = spark_df_final.select(*df_schema_sql.columns)
                print(f"  Tabela existente localizada. Alinhando colunas e fazendo Append...")
            except Exception:
                # Se a tabela não existir, o Spark cria ela do zero com o schema exato do DataFrame
                print(f"  Criando nova tabela diferenciada: [squad2].[{TABELA_SQL}]...")
                spark_df_aligned = spark.createDataFrame(df_novos_dados)
            
            # Gravação final com segurança por colchetes
            spark_df_aligned.write \
                .format("sqlserver") \
                .options(**SQL_OPTIONS) \
                .option("dbtable", f"[squad2].[{TABELA_SQL}]") \
                .mode("append") \
                .save()
            
            # Atualiza o arquivo de controle JSON
            arquivos_atuais = set(df_novos_dados['bronze_source_file'].unique())
            todos_processados = list(processados.union(arquivos_atuais))
            file_client.upload_data(json.dumps(todos_processados), overwrite=True)
            
            print(f" SUCESSO! Dados persistidos em squad2.{TABELA_SQL}!")

except Exception as e:
    print(f" Erro no processamento: {str(e)}")
    raise